
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #02A36F; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lab</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Build a Complete Medallion Pipeline</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Build your own Bronze, Silver, and Gold tables with different transformations and a different aggregation.</div>
</div>

</div>

## REQUIRED — SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before running this notebook, confirm your compute environment at the top-right of the notebook.

- Click the compute dropdown and select **Serverless** (the default option).
- If you do not see Serverless available, contact your workspace administrator.

**Note:** This notebook was developed and tested on **Serverless compute**. Other compute options may work but are not guaranteed to behave the same.
  </div>
</div>

### Setup
Run the cell below to configure your environment.

In [0]:
%run ./Includes/Classroom-Setup-1

### Instructions

In Lesson 12, you built a full Medallion pipeline where Silver uppercased the `Role` column and Gold counted employees by role. Now you'll build your own pipeline with different transformations:
- **Bronze:** Same raw ingestion from CSV files
- **Silver:** Uppercase the `Country` column (not Role) and add a row number
- **Gold:** Count employees by **country** instead of by role

Your goal: create **`practice_bronze`**, **`practice_silver`**, and **`country_count_gold`** tables.

---
## Part 1: Bronze and Silver

### Task 1: Create a Bronze table

Create an empty table called `practice_bronze` with columns: `ID` (INT), `FirstName` (STRING), `Country` (STRING), `Role` (STRING). Then load all CSV files from the volume using COPY INTO.

In [0]:
%sql
-- TODO: Create the empty practice_bronze table
<FILL_IN>

<details>
<summary>Hint</summary>

<pre><code>CREATE TABLE IF NOT EXISTS practice_bronze (
  ID INT,
  FirstName STRING,
  Country STRING,
  Role STRING
);</code></pre>

</details>

In [0]:
# TODO: Load data into practice_bronze with COPY INTO
result = spark.sql("""
    COPY INTO <FILL_IN>
    FROM '<FILL_IN>'
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
""")
result.display()

<details>
<summary>Hint</summary>

<pre><code>result = spark.sql(f"""
    COPY INTO practice_bronze
    FROM '/Volumes/{my_catalog}/{my_schema}/myfiles/'
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
""")
result.display()</code></pre>

</details>

Verify Bronze has 6 rows.

In [0]:
%sql
SELECT * FROM practice_bronze;

### Task 2: Create a Silver table with transformations

Create `practice_silver` from `practice_bronze` with these transformations:
- Uppercase the **Country** column (not Role this time)
- Add a **processed_timestamp** column using `current_timestamp()`
- Add a **row_number** column using the `ROW_NUMBER()` window function ordered by ID

In [0]:
%sql
-- TODO: Create practice_silver with the transformations described above
<FILL_IN>

<details>
<summary>Hint</summary>

<pre><code>CREATE OR REPLACE TABLE practice_silver AS
SELECT
  ID,
  FirstName,
  UPPER(Country) AS Country,
  Role,
  current_timestamp() AS processed_timestamp,
  ROW_NUMBER() OVER (ORDER BY ID) AS row_number
FROM practice_bronze;</code></pre>

</details>

### Task 3: Verify your Silver table

In [0]:
%sql
-- TODO: Query practice_silver
<FILL_IN>

<details>
<summary>Hint</summary>

<pre><code>SELECT * FROM practice_silver;</code></pre>

</details>

You should see 6 rows with:
- **Country** in uppercase (e.g., `UNITED STATES` instead of `United States`)
- A **processed_timestamp** showing when the transformation ran
- A **row_number** from 1 to 6

### Task 4: Compare Bronze and Silver

Run the query below to see the column differences between your Bronze and Silver tables.

In [0]:
%sql
SELECT 'practice_bronze' AS table_name, COUNT(*) AS column_count
FROM information_schema.columns
WHERE table_schema = my_schema AND table_name = 'practice_bronze'
UNION ALL
SELECT 'practice_silver', COUNT(*)
FROM information_schema.columns
WHERE table_schema = my_schema AND table_name = 'practice_silver';

Bronze has **4 columns** (raw data). Silver has **6 columns** (added `processed_timestamp` and `row_number`). The transformation added structure and audit information on top of the raw data.

---
## Part 2: Gold and Governance

### Task 5: Create a temp view for the aggregation

Create a temp view called `temp_country_counts` that counts employees by `Country` from `practice_silver`.

In [0]:
%sql
-- TODO: Create a temp view that counts employees by Country
<FILL_IN>

<details>
<summary>Hint</summary>

<pre><code>CREATE OR REPLACE TEMP VIEW temp_country_counts AS
SELECT
  Country,
  COUNT(*) AS TotalEmployees
FROM practice_silver
GROUP BY Country;</code></pre>

</details>

Verify the aggregation looks right.

In [0]:
%sql
SELECT * FROM temp_country_counts;

### Task 6: Create the Gold table and load it

Create a table called `country_count_gold` with columns `Country` (STRING) and `TotalEmployees` (INT), then use `INSERT OVERWRITE` to populate it from your temp view.

In [0]:
%sql
-- TODO: Create the Gold table and load it
<FILL_IN>

<details>
<summary>Hint</summary>

<pre><code>CREATE TABLE IF NOT EXISTS country_count_gold (
  Country STRING,
  TotalEmployees INT
);

INSERT OVERWRITE country_count_gold
SELECT * FROM temp_country_counts;</code></pre>

</details>

### Task 7: Query the Gold table

In [0]:
%sql
-- TODO: Query country_count_gold
<FILL_IN>

<details>
<summary>Hint</summary>

<pre><code>SELECT * FROM country_count_gold;</code></pre>

</details>

You should see one row per country with the count of employees from each. This answers a different business question than the lesson's Gold table, but both are sourced from the same Silver layer pattern.

### Task 8: Explore governance

Open Catalog Explorer and navigate to `country_count_gold`. Check the **Lineage** tab to see if you can trace it back to `practice_silver` and `practice_bronze`.


<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="margin-top: 10px; padding: 18px 24px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px;">Nice work! You just:</div>
    <ul style="padding-left: 20px; margin: 0;">
      <li>Built a Bronze table with raw data using <code>COPY INTO</code></li>
      <li>Transformed it into a Silver table with <code>UPPER()</code>, timestamps, and <code>ROW_NUMBER()</code></li>
      <li>Created a Gold aggregation table counting employees by country</li>
      <li>Used the <code>INSERT OVERWRITE</code> pattern for refreshable Gold tables</li>
      <li>Explored lineage to trace data from Bronze through Silver to Gold</li>
    </ul>
  </div>
</div>
</div>


<!-- CHECKPOINT: Lesson 12 -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #1B5162; color: white; border-radius: 8px; padding: 24px 28px; text-align: center;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Checkpoint</div>
  <div style="font-size: 20pt; font-weight: 700;">What You've Done So Far</div>
</div>

<div style="margin-top: 16px; padding: 20px 24px; background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.7;">
    <p>Over Lessons 1 through 6, you:</p>
    <ul style="padding-left: 20px; margin: 8px 0;">
      <li>Navigated Unity Catalog and worked with volumes, schemas, and tables</li>
      <li>Created UC tables from CSV files using multiple ingestion methods</li>
      <li>Modified data with DML and explored version history with time travel</li>
      <li>Built a complete Medallion Architecture pipeline (Bronze → Silver → Gold)</li>
      <li>Explored governance features: lineage, permissions, and insights</li>
    </ul>
  </div>
</div>

<div style="margin-top: 16px; padding: 16px 20px; background: #F8F9FC; border-left: 4px solid #1B5162; border-radius: 6px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <strong>Quick self-check:</strong> Could you explain to a teammate why data goes through three layers instead of loading it straight into a final table?
  </div>
</div>



&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>